# Kaggle — House Prices : Advanced Regression Techniques

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
test_ids = test["Id"]  # Id du test, nécessaires pour la soumission

train.shape, test.shape

((1460, 81), (1459, 80))

## 1. Préparation des données

In [2]:
train = train.reset_index(drop=True)

# Cible passée en log (la métrique Kaggle est un RMSE sur le log du prix)
y = np.log1p(train["SalePrice"])

# Features uniquement : on retire l'identifiant et la cible
X_train = train.drop(columns=["Id", "SalePrice"])
X_test  = test.drop(columns=["Id"])

# Concaténation train + test ; n_train sert à re-séparer plus tard
n_train = len(X_train)
all_X = pd.concat([X_train, X_test], axis=0, ignore_index=True)

In [3]:
all_X.shape  # attendu : (2919, 79)

(2919, 79)

## 2. Valeurs manquantes

In [4]:
# Colonnes qui contiennent au moins un NaN
cols_na = all_X.columns[all_X.isnull().any()]
df_null = all_X[cols_na]

# Nombre et pourcentage de NaN, triés par ordre décroissant
na = df_null.isnull().sum().sort_values(ascending=False)
na_pct = (na / len(all_X) * 100).round(1)
pd.DataFrame({"nb_NaN": na, "pct": na_pct})

,nb_NaN,pct
PoolQC,2909,99.7
MiscFeature,2814,96.4
Alley,2721,93.2
Fence,2348,80.4
MasVnrType,1766,60.5
FireplaceQu,1420,48.6
LotFrontage,486,16.6
GarageQual,159,5.4
GarageYrBlt,159,5.4
GarageCond,159,5.4


### 2.1 `PoolQC` — qualité de la piscine (99,7 % de NaN)

D'après `data_description.txt`, `NA` = pas de piscine. On vérifie que les NaN correspondent aux maisons avec `PoolArea == 0`.

In [5]:
# Seulement 10 piscines ont une qualité renseignée
all_X['PoolQC'].value_counts()

PoolQC
Ex    4
Gd    4
Fa    2
Name: count, dtype: int64

**Test de cohérence `PoolQC` × `PoolArea`**

In [6]:
# Maisons sans piscine
(all_X['PoolArea'] == 0).sum()

np.int64(2906)

In [7]:
# Pas de piscine ET pas de qualité -> cohérent
((all_X['PoolArea'] == 0) & (all_X['PoolQC'].isnull())).sum()

np.int64(2906)

In [8]:
# Piscine présente MAIS qualité manquante -> vraies valeurs manquantes
((all_X['PoolArea'] > 0) & (all_X['PoolQC'].isnull())).sum()

np.int64(3)

In [9]:
# Pas de piscine MAIS qualité renseignée -> incohérence
((all_X['PoolArea'] == 0) & (all_X['PoolQC'].notnull())).sum()

np.int64(0)

In [10]:
# Piscine présente ET qualité renseignée -> cohérent
((all_X['PoolArea'] > 0) & (all_X['PoolQC'].notnull())).sum()

np.int64(10)

**Résultat**

| | `PoolQC` vide | `PoolQC` renseigné |
|---|---|---|
| `PoolArea == 0` | 2 906 | 0 |
| `PoolArea > 0` | 3 | 10 |

- 2 906 NaN = pas de piscine.
- 3 piscines sans qualité = vraies valeurs manquantes.
- Aucune incohérence dans l'autre sens.

In [11]:
# Les 3 piscines SANS qualité : train ou test ?
mask = (all_X["PoolArea"] > 0) & (all_X["PoolQC"].isnull())
idx = all_X[mask].index

print(idx)
print(["train" if i < n_train else "test" for i in idx])

Index([2420, 2503, 2599], dtype='int64')
['test', 'test', 'test']


In [12]:
# Les 10 piscines AVEC qualité : train ou test ?
mask = (all_X['PoolArea'] > 0) & (all_X['PoolQC'].notnull())
idx = all_X[mask].index

print(idx)
print(["train" if i < n_train else "test" for i in idx])

Index([197, 810, 1170, 1182, 1298, 1386, 1423, 1974, 2573, 2710], dtype='int64')
['train', 'train', 'train', 'train', 'train', 'train', 'train', 'test', 'test', 'test']


**Conclusion**

- Les 3 piscines sans qualité sont toutes dans le test.
- Seulement 7 piscines notées dans le train sur 1 460 maisons : trop peu pour que le modèle apprenne l'effet de la qualité.
- `PoolArea` contient déjà l'information sur la présence de piscine.

**Décision : suppression de `PoolQC`.**

In [13]:
# Suppression de PoolQC (trop peu d'exemples notés dans le train)
all_X = all_X.drop(columns=["PoolQC"])
all_X.shape  # attendu : (2919, 78)

(2919, 78)

### 2.2 `MiscFeature` — équipement divers (96,4 % de NaN)

D'après `data_description.txt`, `NA` = aucun équipement, et `MiscVal` = valeur en $ de cet équipement. On vérifie que les NaN correspondent aux maisons avec `MiscVal == 0`.

In [23]:
# Répartition des équipements renseignés
all_X['MiscFeature'].value_counts()

MiscFeature
Shed    95
Gar2     5
Othr     4
TenC     1
Name: count, dtype: int64

In [22]:
# Nombre total d'équipements renseignés
all_X['MiscFeature'].value_counts().sum()

np.int64(105)

**Test de cohérence `MiscFeature` × `MiscVal`**

In [15]:
# Maisons sans valeur d'équipement
(all_X['MiscVal'] == 0).sum()

np.int64(2816)

In [17]:
# Pas de valeur ET pas d'équipement -> cohérent
((all_X['MiscVal'] == 0) & (all_X['MiscFeature'].isnull())).sum()

np.int64(2813)

In [19]:
# Valeur renseignée MAIS équipement manquant -> incohérence
((all_X['MiscVal'] > 0) & (all_X['MiscFeature'].isnull())).sum()

np.int64(1)

In [20]:
# Équipement présent MAIS valeur à 0
((all_X['MiscVal'] == 0) & (all_X['MiscFeature'].notnull())).sum()

np.int64(3)

In [21]:
# Équipement présent ET valeur renseignée -> cohérent
((all_X['MiscVal'] > 0) & (all_X['MiscFeature'].notnull())).sum()

np.int64(102)

**Résultat**

| | `MiscFeature` vide | `MiscFeature` renseigné |
|---|---|---|
| `MiscVal == 0` | 2 813 | 3 |
| `MiscVal > 0` | 1 | 102 |

- 2 813 NaN = aucun équipement.
- 1 maison avec une valeur mais sans équipement, 3 équipements avec une valeur à 0.

In [29]:
# Répartition des équipements par catégorie, entre train et test
condition = (all_X['MiscVal'] > 0) & (all_X['MiscFeature'].notnull())
sub = all_X.loc[condition, ['MiscFeature']].copy()
sub['origine'] = ["train" if i < n_train else "test" for i in sub.index]
pd.crosstab(sub['MiscFeature'], sub['origine'], margins=True)

origine,test,train,All
MiscFeature,,,
Gar2,3,2,5
Othr,2,1,3
Shed,45,48,93
TenC,0,1,1
All,50,52,102


**Conclusion**

- Répartition train / test d'environ 50/50 (52 / 50).
- Dans le train, seule `Shed` a assez d'exemples (48) ; `Gar2`, `Othr` et `TenC` en ont 1 ou 2 : le modèle ne peut rien en apprendre.
- `MiscVal` porte déjà l'information utile (présence d'un équipement et sa valeur) ; la catégorie apporte très peu en plus.
- Garder `MiscFeature` imposerait un encodage pour un gain quasi nul.

**Décision : suppression de `MiscFeature`, on conserve `MiscVal`.**

In [ ]:
# Suppression de MiscFeature (MiscVal conserve l'information utile)
all_X = all_X.drop(columns=["MiscFeature"])
all_X.shape  # attendu : (2919, 77)

### 2.3 `Alley` — type d'accès par allée (93,2 % de NaN)